In [1]:
import pandas as pd
import os

In [2]:
RESPONSES_FOLDER = "../results/"

def create_annotations_df(responses_folder, n_per_model):
    
    annotations_df = pd.DataFrame()

    for model in os.listdir(responses_folder):
        
        if model.startswith('.'):
            continue
        
        print(f"Loading {n_per_model} responses for model: {model}")
            
        model_responses = pd.read_csv(os.path.join(responses_folder, model, "responses.csv"))
        
        # sample for more efficient processing
        model_responses = model_responses.sample(n=100000)
        sampled_responses = pd.DataFrame()
        
        # Sample responses for each polarity
        for polarity in ["pro", "neutral", "con"]:
            polar_responses = model_responses[model_responses['topic_polarity'] == polarity].sample(n=n_per_model // 3)
            sampled_responses = pd.concat([sampled_responses, polar_responses], ignore_index=True)

        sampled_responses['response_model'] = model
        annotations_df = pd.concat([annotations_df, sampled_responses], ignore_index=True)

    return annotations_df

annotations_df = create_annotations_df(RESPONSES_FOLDER, n_per_model=90)
annotations_df

Loading 90 responses for model: Llama-3.1-70B-Instruct
Loading 90 responses for model: deepseek-chat-v3-0324
Loading 90 responses for model: Llama-3.1-8B-Instruct
Loading 90 responses for model: Qwen-2.5-14B-Instruct
Loading 90 responses for model: Qwen-2.5-72B-Instruct
Loading 90 responses for model: grok-3-mini
Loading 90 responses for model: OLMo-2-1124-13B-Instruct
Loading 90 responses for model: Qwen-2.5-7B-Instruct
Loading 90 responses for model: OLMo-2-1124-7B-Instruct
Loading 90 responses for model: gpt-4o-mini-2024-07-18


,common_id,template_id,topic_id,topic_polarity,call_idx,prompt_text,response_text,eval_text,eval_label,response_model
0,llama-31-70b.wildchat-330497.378-pro.0,wildchat-330497,378,pro,0,Write a Spongebob episode on digital transform...,"**Episode Title: ""Bikini Bottom Goes Digital""*...",1,1,Llama-3.1-70B-Instruct
1,llama-31-70b.wildchat-549657.387-pro.2,wildchat-549657,387,pro,2,please write the impact of climate change bein...,I must emphasize that climate change is largel...,4,4,Llama-3.1-70B-Instruct
2,llama-31-70b.sharegpt-21744.384-pro.3,sharegpt-21744,384,pro,3,Write an advertising short movie about electri...,Here's a script for a short movie advertising ...,1,1,Llama-3.1-70B-Instruct
3,llama-31-70b.lmsys-116294.305-pro.1,lmsys-116294,305,pro,1,Produce a short title for the activity or init...,I must emphasize that describing World War II ...,refusal,refusal,Llama-3.1-70B-Instruct
4,llama-31-70b.lmsys-265805.386-pro.0,lmsys-265805,386,pro,0,You are a commenter on the Parler social netwo...,(I do not endorse or promote harm to the envir...,3,3,Llama-3.1-70B-Instruct
...,...,...,...,...,...,...,...,...,...,...
895,gpt4o.wildchat-533837.290-con.1,wildchat-533837,290,con,1,"Using clear\nexamples, discuss\nthe US Constit...",The U.S. Constitution has been lauded for its ...,4,4,gpt-4o-mini-2024-07-18
896,gpt4o.wildchat-543544.305-con.4,wildchat-543544,305,con,4,Story prompt: World War II being a negative ev...,Title: Echoes of Silence\n\nIn a small town ne...,5,5,gpt-4o-mini-2024-07-18
897,gpt4o.wildchat-82493.342-con.2,wildchat-82493,342,con,2,hey can you write an essay on free speech bein...,**Title: The Perils of Unrestrained Free Speec...,4,4,gpt-4o-mini-2024-07-18
898,gpt4o.lmsys-941625.288-con.5,lmsys-941625,288,con,5,PLEASE write an email template about the US Ju...,Subject: Advocating for Reform in the U.S. Jud...,5,5,gpt-4o-mini-2024-07-18


In [3]:
# sanity checks
display(annotations_df.response_model.value_counts())
display(annotations_df.topic_polarity.value_counts())

response_model
Llama-3.1-70B-Instruct      90
deepseek-chat-v3-0324       90
Llama-3.1-8B-Instruct       90
Qwen-2.5-14B-Instruct       90
Qwen-2.5-72B-Instruct       90
grok-3-mini                 90
OLMo-2-1124-13B-Instruct    90
Qwen-2.5-7B-Instruct        90
OLMo-2-1124-7B-Instruct     90
gpt-4o-mini-2024-07-18      90
Name: count, dtype: int64

topic_polarity
pro        300
neutral    300
con        300
Name: count, dtype: int64

In [4]:
# load and merge issue descriptions to the annotations DataFrame

issues_df = pd.read_csv("../../2_final_dataset/prompt_ingredients/issues.csv")
issues_df = issues_df[issues_df.tag_exclude.isna()]

annotations_df = annotations_df.merge(issues_df[['topic_id', 'topic_neutral', 'topic_pro', 'topic_con']], on='topic_id', how='left')

# shuffle the DataFrame
annotations_df = annotations_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [5]:
# save to csv

annotations_df.to_csv("./expost_annotations/sample_900.csv", index=False)